# MouseTrap Scout — Edge Classifier Training (v2)

Trains an int8 MobileNet v2 α=0.35 @ 96×96 classifier for on-device rodent detection on the XIAO ESP32-S3 Sense.

**3 classes:** `rodent`, `person_or_pet`, `other`

## Data sources (all used)
| Source | Class | Images | Why |
|--------|-------|--------|-----|
| Channel Islands Camera Traps (LILA) | rodent | ~5K sampled from 82K | Real camera trap images with COCO bboxes |
| NZ TrailCams (LILA) | rodent | ~3K sampled from 1.2M | Massive mouse/rat dataset, camera trap style |
| Roboflow rodent datasets | rodent | ~2K | Pre-annotated, diverse angles |
| iNaturalist Rodentia | rodent | ~2K | Nature photos for diversity |
| COCO 2017 val | person_or_pet | ~3K | Person, cat, dog images |
| COCO 2017 val | other | ~3K | Indoor scenes, objects, backgrounds |
| Your scout images (optional) | mixed | varies | Real-world corrections from `export-training-data.sh` |

**Total: ~18K images.** Runtime: ~30–40 min on Colab free tier (T4 GPU).

## 0. Setup

In [ ]:
import tensorflow as tf
import numpy as np
import os, json, shutil, hashlib, urllib.request, zipfile, tarfile, io, time, random
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
import requests

print('TF version:', tf.__version__)
print('GPU available:', len(tf.config.list_physical_devices('GPU')) > 0)

SEED = 1337
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

DATA_ROOT = Path('/content/mousetrap_data')
for c in ['rodent', 'person_or_pet', 'other']:
    (DATA_ROOT / c).mkdir(parents=True, exist_ok=True)

def download_image(url, dest_path, timeout=15):
    """Download a single image. Returns True on success."""
    try:
        r = requests.get(url, timeout=timeout)
        if r.status_code == 200 and len(r.content) > 1000:
            dest_path.write_bytes(r.content)
            return True
    except Exception:
        pass
    return False

def download_batch(url_dest_pairs, desc='Downloading', max_workers=8):
    """Download images in parallel. Returns count of successes."""
    downloaded = 0
    total = len(url_dest_pairs)
    with ThreadPoolExecutor(max_workers=max_workers) as pool:
        futures = {pool.submit(download_image, url, dest): (url, dest)
                   for url, dest in url_dest_pairs}
        for f in as_completed(futures):
            if f.result():
                downloaded += 1
                if downloaded % 200 == 0:
                    print(f'  {desc}: {downloaded}/{total}')
    print(f'  {desc}: {downloaded}/{total} done')
    return downloaded

print('Setup complete. Data root:', DATA_ROOT)

## 1a. Channel Islands Camera Traps (LILA)

82,914 rodent images with COCO bounding boxes. Real camera trap images — the best training data for our use case. We sample 5K to keep training manageable.

In [ ]:
CI_TARGET = 5000
CI_META_URL = 'https://lilablobssc.blob.core.windows.net/channel-islands-camera-traps/channel-islands-camera-traps.json'
CI_IMAGE_BASE = 'https://lilablobssc.blob.core.windows.net/channel-islands-camera-traps/'
CI_META_PATH = Path('/content/ci_metadata.json')
RODENT_DIR = DATA_ROOT / 'rodent'

# Download metadata (~50MB JSON)
if not CI_META_PATH.exists():
    print('Downloading Channel Islands metadata...')
    urllib.request.urlretrieve(CI_META_URL, CI_META_PATH)
    print(f'Metadata downloaded: {CI_META_PATH.stat().st_size / 1024 / 1024:.1f} MB')
else:
    print('Channel Islands metadata already downloaded')

# Parse and find rodent images
print('Parsing metadata...')
with open(CI_META_PATH) as f:
    ci_data = json.load(f)

# Build category lookup
ci_cat_map = {c['id']: c['name'].lower() for c in ci_data['categories']}
rodent_cat_ids = {cid for cid, name in ci_cat_map.items()
                  if any(r in name for r in ['rodent', 'mouse', 'rat', 'squirrel', 'vole', 'gopher'])}
print(f'Rodent categories: {[(cid, ci_cat_map[cid]) for cid in rodent_cat_ids]}')

# Find image IDs with rodent annotations
rodent_img_ids = set()
for ann in ci_data.get('annotations', []):
    if ann['category_id'] in rodent_cat_ids:
        rodent_img_ids.add(ann['image_id'])
print(f'Images with rodent annotations: {len(rodent_img_ids)}')

# Build image ID -> filename lookup
img_lookup = {img['id']: img['file_name'] for img in ci_data['images']}

# Sample and download
rodent_ids_list = list(rodent_img_ids)
random.shuffle(rodent_ids_list)
sampled = rodent_ids_list[:CI_TARGET]

pairs = []
for img_id in sampled:
    fname = img_lookup.get(img_id)
    if not fname:
        continue
    dest = RODENT_DIR / f'ci_{img_id}.jpg'
    if not dest.exists():
        pairs.append((CI_IMAGE_BASE + fname, dest))

if pairs:
    download_batch(pairs, desc='Channel Islands')
else:
    print('Channel Islands images already downloaded')

ci_count = len(list(RODENT_DIR.glob('ci_*.jpg')))
print(f'Channel Islands rodent images: {ci_count}')

## 1b. NZ TrailCams (LILA)

~1.2M mouse + ~137K rat images from New Zealand trail cameras. Massive dataset — we sample 3K.

In [ ]:
NZ_TARGET = 3000
NZ_META_URL = 'https://lilablobssc.blob.core.windows.net/nz-trailcams/nz-trailcams.json'
NZ_IMAGE_BASE = 'https://lilablobssc.blob.core.windows.net/nz-trailcams/'
NZ_META_PATH = Path('/content/nz_metadata.json')

# Download metadata (~large JSON)
if not NZ_META_PATH.exists():
    print('Downloading NZ TrailCams metadata (this may take a minute)...')
    urllib.request.urlretrieve(NZ_META_URL, NZ_META_PATH)
    print(f'Metadata downloaded: {NZ_META_PATH.stat().st_size / 1024 / 1024:.1f} MB')
else:
    print('NZ TrailCams metadata already downloaded')

print('Parsing metadata...')
with open(NZ_META_PATH) as f:
    nz_data = json.load(f)

# Build category lookup
nz_cat_map = {c['id']: c['name'].lower() for c in nz_data['categories']}
nz_rodent_cat_ids = {cid for cid, name in nz_cat_map.items()
                     if any(r in name for r in ['mouse', 'rat', 'rodent'])}
print(f'Rodent categories: {[(cid, nz_cat_map[cid]) for cid in nz_rodent_cat_ids]}')

# Find image IDs with rodent annotations
nz_rodent_img_ids = set()
for ann in nz_data.get('annotations', []):
    if ann['category_id'] in nz_rodent_cat_ids:
        nz_rodent_img_ids.add(ann['image_id'])
print(f'NZ images with rodent annotations: {len(nz_rodent_img_ids)}')

nz_img_lookup = {img['id']: img['file_name'] for img in nz_data['images']}

# Sample and download
nz_sampled = random.sample(list(nz_rodent_img_ids), min(NZ_TARGET, len(nz_rodent_img_ids)))

pairs = []
for img_id in nz_sampled:
    fname = nz_img_lookup.get(img_id)
    if not fname:
        continue
    dest = RODENT_DIR / f'nz_{img_id}.jpg'
    if not dest.exists():
        pairs.append((NZ_IMAGE_BASE + fname, dest))

if pairs:
    download_batch(pairs, desc='NZ TrailCams')
else:
    print('NZ TrailCams images already downloaded')

nz_count = len(list(RODENT_DIR.glob('nz_*.jpg')))
print(f'NZ TrailCams rodent images: {nz_count}')

## 1c. Roboflow rodent datasets

Several pre-annotated rodent datasets from Roboflow Universe. Downloads via their export API (no API key needed for public datasets).

In [ ]:
# Roboflow public datasets - download as zip via their export links
ROBOFLOW_DATASETS = [
    # (name, download_url) - classification format exports
    ('rodent_detection', 'https://universe.roboflow.com/ds/guna-nadar-ovhuz/rodent_detection-1qmwb'),
    ('detect_rats', 'https://universe.roboflow.com/ds/guna-nadar-ovhuz/detect-rats'),
]

# Roboflow requires pip install for easiest download
try:
    from roboflow import Roboflow
    HAS_ROBOFLOW = True
except ImportError:
    print('Installing roboflow package...')
    import subprocess
    subprocess.run(['pip', 'install', '-q', 'roboflow'], check=True)
    try:
        from roboflow import Roboflow
        HAS_ROBOFLOW = True
    except ImportError:
        HAS_ROBOFLOW = False
        print('Roboflow install failed — skipping Roboflow datasets')

rf_count = 0
if HAS_ROBOFLOW:
    # Download public datasets without API key
    # These are downloaded as image folders
    RF_DIR = Path('/content/roboflow_rodents')
    RF_DIR.mkdir(exist_ok=True)

    try:
        rf = Roboflow(api_key='')  # Public datasets don't need a key

        # Download rodent_detection dataset
        print('Attempting Roboflow download (may need API key for some datasets)...')
        project = rf.workspace('guna-nadar-ovhuz').project('rodent_detection-1qmwb')
        version = project.version(1)
        dataset = version.download('folder', location=str(RF_DIR / 'rodent_detection'))
    except Exception as e:
        print(f'Roboflow download failed (expected without API key): {e}')
        print('Skipping Roboflow — the other sources provide enough rodent data.')

    # Copy any downloaded images
    for img in RF_DIR.rglob('*.jpg'):
        dest = RODENT_DIR / f'rf_{img.stem}.jpg'
        if not dest.exists():
            shutil.copy(img, dest)
            rf_count += 1

print(f'Roboflow rodent images: {rf_count}')

## 1d. iNaturalist Rodentia (supplementary)

Nature photography of rodents — different style from camera traps but adds diversity.

In [ ]:
INAT_TARGET = 2000
RODENTIA_TAXON_ID = 43094

def fetch_inat_observations(taxon_id, per_page=200, max_count=2000):
    page = 1
    count = 0
    while count < max_count:
        url = (
            'https://api.inaturalist.org/v1/observations'
            f'?taxon_id={taxon_id}&photos=true&quality_grade=research'
            f'&license=cc-by,cc-by-nc,cc0&per_page={per_page}&page={page}'
            '&order_by=random'
        )
        r = requests.get(url, timeout=30)
        if r.status_code != 200:
            break
        results = r.json().get('results', [])
        if not results:
            break
        for obs in results:
            for photo in obs.get('photos', []):
                url_med = photo.get('url', '').replace('square', 'medium')
                if url_med:
                    yield (obs['id'], url_med)
                    count += 1
                    if count >= max_count:
                        return
        page += 1
        time.sleep(1.0)

existing_inat = len(list(RODENT_DIR.glob('inat_*.jpg')))
if existing_inat >= INAT_TARGET * 0.9:
    print(f'Already have {existing_inat} iNaturalist images')
else:
    print(f'Downloading up to {INAT_TARGET} iNaturalist rodent images...')
    pairs = []
    for obs_id, photo_url in fetch_inat_observations(RODENTIA_TAXON_ID, max_count=INAT_TARGET * 2):
        if len(pairs) >= INAT_TARGET:
            break
        dest = RODENT_DIR / f'inat_{obs_id}_{len(pairs)}.jpg'
        if not dest.exists():
            pairs.append((photo_url, dest))
    download_batch(pairs, desc='iNaturalist')

inat_count = len(list(RODENT_DIR.glob('inat_*.jpg')))
print(f'iNaturalist rodent images: {inat_count}')

## 2. COCO 2017 for person_or_pet + other classes

In [ ]:
COCO_DIR = Path('/content/coco2017')
COCO_DIR.mkdir(parents=True, exist_ok=True)

COCO_IMAGES_URL = 'http://images.cocodataset.org/zips/val2017.zip'
COCO_ANNOTATIONS_URL = 'http://images.cocodataset.org/annotations/annotations_trainval2017.zip'

def download_and_extract(url, dest_dir, marker_file):
    if (dest_dir / marker_file).exists():
        print(f'{marker_file} already present, skipping')
        return
    print(f'Downloading {url} ...')
    local = dest_dir / url.split('/')[-1]
    with urllib.request.urlopen(url) as resp, open(local, 'wb') as f:
        shutil.copyfileobj(resp, f)
    print(f'Extracting...')
    with zipfile.ZipFile(local) as z:
        z.extractall(dest_dir)
    local.unlink()

download_and_extract(COCO_IMAGES_URL, COCO_DIR, 'val2017')
download_and_extract(COCO_ANNOTATIONS_URL, COCO_DIR, 'annotations')

# Parse annotations
with open(COCO_DIR / 'annotations' / 'instances_val2017.json') as f:
    coco = json.load(f)

cat_by_id = {c['id']: c['name'] for c in coco['categories']}
img_by_id = {img['id']: img['file_name'] for img in coco['images']}

from collections import defaultdict
img_categories = defaultdict(set)
for ann in coco['annotations']:
    img_categories[ann['image_id']].add(cat_by_id[ann['category_id']])

TARGET_PET_PERSON = {'person', 'cat', 'dog'}
ANY_ANIMAL = {'person', 'cat', 'dog', 'bird', 'horse', 'sheep', 'cow', 'bear',
              'elephant', 'giraffe', 'zebra'}

pet_person_files = []
other_files = []

for img_id, categories in img_categories.items():
    fname = img_by_id.get(img_id)
    if not fname: continue
    path = COCO_DIR / 'val2017' / fname
    if not path.exists(): continue
    if categories & TARGET_PET_PERSON:
        pet_person_files.append(path)
    elif not (categories & ANY_ANIMAL):
        other_files.append(path)

# Add unannotated images as 'other'
annotated_ids = set(img_categories.keys())
for img in coco['images']:
    if img['id'] not in annotated_ids:
        path = COCO_DIR / 'val2017' / img['file_name']
        if path.exists():
            other_files.append(path)

random.shuffle(pet_person_files)
random.shuffle(other_files)

PP_DIR = DATA_ROOT / 'person_or_pet'
OTHER_DIR = DATA_ROOT / 'other'

for i, src in enumerate(pet_person_files[:3000]):
    dst = PP_DIR / f'coco_pp_{i}.jpg'
    if not dst.exists(): shutil.copy(src, dst)

for i, src in enumerate(other_files[:3000]):
    dst = OTHER_DIR / f'coco_other_{i}.jpg'
    if not dst.exists(): shutil.copy(src, dst)

print(f'COCO person_or_pet: {len(list(PP_DIR.glob("*.jpg")))}')
print(f'COCO other: {len(list(OTHER_DIR.glob("*.jpg")))}')

## 2b. Optional: Upload your own scout images

If you've exported training data from your scout using `Server/scripts/export-training-data.sh`, upload the tarball here. Images will be mixed into the appropriate class folders.

**Skip this cell** if you don't have real data yet.

In [ ]:
# Uncomment and run to upload your scout training data
# from google.colab import files
# uploaded = files.upload()  # Select training-data-*.tar.gz
# for name, data in uploaded.items():
#     with tarfile.open(fileobj=io.BytesIO(data)) as tar:
#         tar.extractall('/content/scout_data')
#     # Copy images into class folders
#     scout_dir = Path('/content/scout_data/images')
#     for cls_dir in scout_dir.iterdir():
#         if cls_dir.is_dir():
#             target = DATA_ROOT / cls_dir.name
#             target.mkdir(exist_ok=True)
#             for img in cls_dir.glob('*.jpg'):
#                 dest = target / f'scout_{img.name}'
#                 if not dest.exists():
#                     shutil.copy(img, dest)
#     print('Scout data imported')
print('(Skipped — uncomment above to upload scout data)')

## 3. Dataset summary

In [ ]:
print('=== Final dataset ===')
total = 0
for cls in ['rodent', 'person_or_pet', 'other']:
    d = DATA_ROOT / cls
    count = len(list(d.glob('*.jpg')))
    total += count
    # Show source breakdown for rodent
    if cls == 'rodent':
        ci = len(list(d.glob('ci_*.jpg')))
        nz = len(list(d.glob('nz_*.jpg')))
        rf = len(list(d.glob('rf_*.jpg')))
        inat = len(list(d.glob('inat_*.jpg')))
        scout = len(list(d.glob('scout_*.jpg')))
        other_src = count - ci - nz - rf - inat - scout
        print(f'  {cls}: {count} total')
        print(f'    Channel Islands: {ci}')
        print(f'    NZ TrailCams:    {nz}')
        print(f'    Roboflow:        {rf}')
        print(f'    iNaturalist:     {inat}')
        if scout: print(f'    Scout (real):    {scout}')
        if other_src: print(f'    Other:           {other_src}')
    else:
        print(f'  {cls}: {count}')
print(f'  TOTAL: {total}')

## 4. Build tf.data pipeline

96×96 RGB, augmentation, 80/20 split.

In [ ]:
IMG_SIZE = 96
BATCH_SIZE = 64
CLASS_NAMES = ['rodent', 'person_or_pet', 'other']
NUM_CLASSES = len(CLASS_NAMES)

train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_ROOT,
    validation_split=0.2,
    subset='training',
    seed=SEED,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_names=CLASS_NAMES,
    shuffle=True,
)
val_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_ROOT,
    validation_split=0.2,
    subset='validation',
    seed=SEED,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_names=CLASS_NAMES,
    shuffle=False,
)

AUTOTUNE = tf.data.AUTOTUNE

augment = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.15),
    tf.keras.layers.RandomZoom(0.15),
    tf.keras.layers.RandomBrightness(0.2),
    tf.keras.layers.RandomContrast(0.2),
])

train_ds = train_ds.map(lambda x, y: (augment(x, training=True), y), num_parallel_calls=AUTOTUNE)
train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)

print(f'Classes: {CLASS_NAMES}')
print(f'Train batches: {tf.data.experimental.cardinality(train_ds).numpy()}')
print(f'Val batches: {tf.data.experimental.cardinality(val_ds).numpy()}')

## 5. Build MobileNet v2 α=0.35

In [ ]:
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    alpha=0.35,
    include_top=False,
    weights='imagenet',
)
base_model.trainable = False

inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = tf.keras.applications.mobilenet_v2.preprocess_input(inputs)
x = base_model(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.3)(x)
outputs = tf.keras.layers.Dense(NUM_CLASSES, activation='softmax')(x)
model = tf.keras.Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)
model.summary()

## 6. Train phase 1 — frozen base, head only

In [ ]:
history1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    verbose=1,
)

## 7. Train phase 2 — unfreeze top layers, fine-tune

In [ ]:
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

history2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    verbose=1,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(
            monitor='val_accuracy', patience=5, restore_best_weights=True
        ),
    ],
)

val_loss, val_acc = model.evaluate(val_ds)
print(f'\nFinal val accuracy: {val_acc:.4f}  val loss: {val_loss:.4f}')

## 8. Int8 quantization

In [ ]:
def representative_dataset():
    for images, _ in val_ds.unbatch().batch(1).take(300):
        yield [tf.cast(images, tf.float32).numpy()]

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

tflite_model = converter.convert()

OUT_DIR = Path('/content/output')
OUT_DIR.mkdir(exist_ok=True)
(OUT_DIR / 'model.tflite').write_bytes(tflite_model)
(OUT_DIR / 'labels.txt').write_text('\n'.join(CLASS_NAMES) + '\n')

print(f'Quantized model size: {len(tflite_model) / 1024:.1f} KB')

## 9. Verify quantized accuracy + confusion matrix

In [ ]:
interpreter = tf.lite.Interpreter(model_content=tflite_model)
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()[0]
output_details = interpreter.get_output_details()[0]

print('Input:', input_details['shape'], input_details['dtype'].__name__,
      f'scale={input_details["quantization"][0]:.4f} zp={input_details["quantization"][1]}')
print('Output:', output_details['shape'], output_details['dtype'].__name__,
      f'scale={output_details["quantization"][0]:.6f} zp={output_details["quantization"][1]}')

in_scale, in_zp = input_details['quantization']

correct = 0
total = 0
per_class_correct = [0] * NUM_CLASSES
per_class_total = [0] * NUM_CLASSES
confusion = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=int)

for images, labels in val_ds.unbatch().batch(1):
    img_float = tf.cast(images, tf.float32).numpy()
    img_int8 = (img_float / in_scale + in_zp).astype(np.int8)
    interpreter.set_tensor(input_details['index'], img_int8)
    interpreter.invoke()
    out = interpreter.get_tensor(output_details['index'])
    pred = int(np.argmax(out[0]))
    true = int(labels[0])
    confusion[true][pred] += 1
    per_class_total[true] += 1
    if pred == true:
        correct += 1
        per_class_correct[true] += 1
    total += 1

print(f'\nQuantized val accuracy: {correct/total:.4f}  ({correct}/{total})')
print('\nPer-class accuracy:')
for i, name in enumerate(CLASS_NAMES):
    if per_class_total[i] > 0:
        acc = per_class_correct[i] / per_class_total[i]
        print(f'  {name:>15}: {acc:.4f}  ({per_class_correct[i]}/{per_class_total[i]})')

print('\nConfusion matrix (rows=truth, cols=predicted):')
header = '                ' + '  '.join(f'{c:>15}' for c in CLASS_NAMES)
print(header)
for i, name in enumerate(CLASS_NAMES):
    row = '  '.join(f'{v:>15}' for v in confusion[i])
    print(f'  {name:>14}  {row}')

# Key metric: rodent recall (how many real rodents did we catch?)
if per_class_total[0] > 0:
    recall = per_class_correct[0] / per_class_total[0]
    precision = per_class_correct[0] / max(1, sum(confusion[:, 0]))
    print(f'\nRodent recall:    {recall:.4f} (miss rate: {1-recall:.4f})')
    print(f'Rodent precision: {precision:.4f} (false positive rate: {1-precision:.4f})')

## 10. Download model

In [ ]:
from google.colab import files
files.download(str(OUT_DIR / 'model.tflite'))
files.download(str(OUT_DIR / 'labels.txt'))
print('Done — check your browser downloads.')
print(f'Deploy: copy model.tflite to scout_arduino/scout-spa/public/')
print(f'Then: make build-fs && make upload-fs')